# FREQ-PRUNE · GoogLeNet · CIFAR-100
**50 base epochs → threshold pruning → 50 fine-tuning epochs.**

Run all cells on a T4 GPU. Drive stores a checkpoint after every epoch, so a disconnected run can resume.
The final cell downloads a ZIP containing both trained models and their reload code; all detailed results stay in Drive.

## Method and assumptions
1. Average **absolute weights across input channels first**, then take one 2D DCT per filter.
2. Square the DCT coefficients and normalize the nine frequency energies.
3. Fit NMF separately per layer with rank 3 and compare normalized fingerprints using cosine similarity ≥ **0.95**.
4. Keep one actual medoid per redundancy group; every pair in a group must pass the similarity checks.
5. Prune each Inception block's 3x3 and 5x5 branch dimensionality reduction convolutions (the 1x1 convolutions), their BatchNorms, and the matching inputs of the subsequent 3x3 or 5x5 convolutions.
6. Preserve residual/concat output widths, skip connections, the stem and classifier; copy surviving trained weights exactly.

There is **no entropy score, positional mask, target pruning ratio, or forced channel alignment**.
This is CIFAR-adapted GoogLeNet: 3×3 stem, stride 1, no initial max-pool, standard Inception modules.
Rank 3 is an experimental choice, not a consequence of the 3×3 kernel size.

**Extra match check:** folded signed-weight cosine must be ≥ 0.20 and relative folded-bias difference ≤ 0.25.
These configurable guards were discussed in the methodology; 0.95 alone is not the whole removal rule.
Complete-link groups prevent a chain of similar pairs from deleting mutually dissimilar filters.

Model selection uses validation accuracy. Each stage completes all 50 epochs, then loads its best validation checkpoint.

## 1. Settings
Use a new `run_name` when changing settings; keep it unchanged to resume this exact experiment.

In [2]:
CONFIG = {
    "run_name": "freqprune_googlenet_c100_avgabs_r3_sim095_seed42_50plus50_v1",
    "seed": 42, "base_epochs": 50, "finetune_epochs": 50,
    "batch_size": 128, "eval_batch_size": 256, "num_workers": 2,
    "base_lr": 0.1, "finetune_lr": 0.01, "warmup_epochs": 5,
    "momentum": 0.9, "weight_decay": 5e-4,
    "nmf_rank": 3, "similarity_threshold": 0.95,
    "nmf_max_iter": 5000, "nmf_retry_max_iter": 20000, "nmf_tolerance": 1e-4,
    "signed_cosine_min": 0.20, "relative_bias_max": 0.25,
    "validation_per_class": 50, "bn_calibration_per_class": 50,
    "amp": True, "resume": True, "save_to_drive": True,
    "download_models": True,
}
assert 0 < CONFIG["similarity_threshold"] <= 1
assert 1 <= CONFIG["nmf_rank"] <= 9

## 2. Libraries and GPU
Keep Colab's existing PyTorch installation and record package versions with the experiment.

In [3]:
import os, sys, json, math, time, copy, random, hashlib, shutil, warnings, inspect
import importlib.util, subprocess
from pathlib import Path
from datetime import datetime, timezone
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
needed = {"scipy": "scipy", "sklearn": "scikit-learn"}
missing = [p for m,p in needed.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
from scipy.fft import dctn
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.stats import binomtest
from sklearn.decomposition import NMF
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits
from IPython.display import display
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then run all again.")
AMP = bool(CONFIG["amp"])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_all(CONFIG["seed"])
plt.rcParams.update({"figure.figsize": (9,4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)

GPU: Tesla T4
PyTorch: 2.11.0+cu128 | torchvision: 0.26.0+cu128


## 3. Save folder and protection against accidental overwrites
Connect Drive if prompted; checkpoints, logs and figures are saved inside this run's own folder.

In [4]:
if CONFIG["save_to_drive"]:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_ROOT = Path("/content/drive/MyDrive/FREQ_PRUNE_Experiments")
else:
    SAVE_ROOT = Path("/content/FREQ_PRUNE_Experiments")
    print("Local runtime files can disappear; download the final archive.")
RUN_DIR = SAVE_ROOT / CONFIG["run_name"]
RUN_DIR.mkdir(parents=True, exist_ok=True)
def write_json(value, path):
    path = Path(path); temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temp, path)
def save_torch(value, path):
    path = Path(path); temp = path.with_name(path.name + ".tmp")
    torch.save(value, temp); os.replace(temp, path)
config_path = RUN_DIR / "config.json"
if config_path.exists():
    assert json.loads(config_path.read_text()) == CONFIG, "Use a new run_name for changed settings."
    assert CONFIG["resume"], "Use a new run_name; this folder already contains a run."
else:
    write_json(CONFIG, config_path)
import scipy, sklearn
write_json({"python": sys.version, "torch": torch.__version__, "torchvision": torchvision.__version__,
            "numpy": np.__version__, "scipy": scipy.__version__, "sklearn": sklearn.__version__,
            "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda,
            "cudnn": torch.backends.cudnn.version()},
           RUN_DIR / ("environment_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + ".json"))
print("Results and checkpoints:", RUN_DIR)

Mounted at /content/drive
Results and checkpoints: /content/drive/MyDrive/FREQ_PRUNE_Experiments/freqprune_googlenet_c100_avgabs_r3_sim095_seed42_50plus50_v1


## 4. CIFAR-100 with a separate validation set
Random crops and flips are used only for training; all accuracy comparisons use identical evaluation transforms.

In [5]:
MEAN, STD = (0.5071,0.4867,0.4408), (0.2675,0.2565,0.2761)
train_transform = transforms.Compose([transforms.RandomCrop(32,padding=4),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize(MEAN,STD)])
eval_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN,STD)])
data_root = "/content/cifar100_data"
all_train = torchvision.datasets.CIFAR100(data_root, train=True, download=True, transform=train_transform)
all_eval = torchvision.datasets.CIFAR100(data_root, train=True, download=False, transform=eval_transform)
targets = np.asarray(all_train.targets)
rng = np.random.default_rng(CONFIG["seed"])
training_indices, validation_indices = [], []
for label in range(100):
    indices = rng.permutation(np.flatnonzero(targets == label))
    validation_indices.extend(indices[:CONFIG["validation_per_class"]])
    training_indices.extend(indices[CONFIG["validation_per_class"]:])
training_indices, validation_indices = np.sort(training_indices), np.sort(validation_indices)
assert len(set(training_indices) & set(validation_indices)) == 0
assert len(training_indices) + len(validation_indices) == 50000
split_path = RUN_DIR / "split_indices.npz"
if split_path.exists():
    old = np.load(split_path)
    assert np.array_equal(old["train"], training_indices) and np.array_equal(old["validation"], validation_indices)
else:
    np.savez_compressed(split_path, train=training_indices, validation=validation_indices)
TRAIN = Subset(all_train, training_indices)
TRAIN_EVAL = Subset(all_eval, training_indices)
VALIDATION = Subset(all_eval, validation_indices)
calibration_indices = np.concatenate([rng.permutation(training_indices[targets[training_indices] == label])
                                     [:CONFIG["bn_calibration_per_class"]] for label in range(100)])
CALIBRATION = Subset(all_eval, calibration_indices)
np.save(RUN_DIR / "calibration_indices.npy", calibration_indices)
def seed_worker(worker_id):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed); random.seed(seed)
def make_loader(dataset, training=False, seed=0):
    return DataLoader(dataset, batch_size=CONFIG["batch_size"] if training else CONFIG["eval_batch_size"],
        shuffle=training, num_workers=CONFIG["num_workers"], pin_memory=True,
        generator=torch.Generator().manual_seed(seed), worker_init_fn=seed_worker)
val_loader = make_loader(VALIDATION)
calibration_loader = make_loader(CALIBRATION)
CLASSES = all_train.classes
write_json({"dataset": "CIFAR100", "train":len(TRAIN), "validation":len(VALIDATION),
            "test_reserved":10000, "classes":CLASSES, "mean":MEAN, "std":STD,
            "split_sha256":hashlib.sha256(training_indices.tobytes()+validation_indices.tobytes()).hexdigest()},
           RUN_DIR / "dataset.json")
print("Train:",len(TRAIN), "| Validation:",len(VALIDATION), "| Test reserved: 10000")
assert len(CLASSES)==100 and len(TRAIN)==45000 and len(VALIDATION)==5000
assert len(CALIBRATION)==5000
assert np.all(np.bincount(targets[validation_indices], minlength=100)==50)

100%|██████████| 169M/169M [41:01<00:00, 68.7kB/s]


Train: 45000 | Validation: 5000 | Test reserved: 10000


## 5. GoogLeNet with independently resizable internal channels
The 3x3 and 5x5 branches in each Inception module have initial 1x1 reductions. These internal bottleneck widths can become narrower.

In [6]:
class BasicConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, padding=0):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=False)
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

class PrunableBranch(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, kernel_size):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, hidden_channels, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(hidden_channels)
        self.relu1 = nn.ReLU(inplace=False)
        self.conv2 = nn.Conv2d(hidden_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu2 = nn.ReLU(inplace=False)
    def forward(self, x):
        out = self.relu1(self.bn1(self.conv1(x)))
        return self.relu2(self.bn2(self.conv2(out)))

class InceptionModule(nn.Module):
    def __init__(self, in_channels, ch1x1, ch3x3red, ch3x3, ch5x5red, ch5x5, pool_proj):
        super().__init__()
        self.branch1 = BasicConv2d(in_channels, ch1x1, kernel_size=1)
        self.branch2 = PrunableBranch(in_channels, ch3x3red, ch3x3, kernel_size=3)
        self.branch3 = PrunableBranch(in_channels, ch5x5red, ch5x5, kernel_size=5)
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            BasicConv2d(in_channels, pool_proj, kernel_size=1)
        )
    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], 1)

class GoogLeNetCIFAR(nn.Module):
    def __init__(self, hidden_widths=None, num_classes=100):
        super().__init__()
        # Standard Inception V1 internal reduction sizes (9 modules * 2 prunable branches = 18 widths)
        ch3x3red = [96, 128, 96, 112, 128, 144, 160, 160, 192]
        ch5x5red = [16,  32, 16,  24,  24,  32,  32,  32,  48]
        default_widths = []
        for a, b in zip(ch3x3red, ch5x5red):
            default_widths.extend([a, b])

        self.hidden_widths = list(default_widths if hidden_widths is None else hidden_widths)
        assert len(self.hidden_widths) == 18 and min(self.hidden_widths) >= 1
        self.num_classes = num_classes

        # CIFAR-adapted stem (3x3 instead of 7x7, no initial maxpool)
        self.stem = BasicConv2d(3, 192, kernel_size=3, padding=1)

        self.inc3a = InceptionModule(192,  64,  self.hidden_widths[0],  128, self.hidden_widths[1],  32,  32)
        self.inc3b = InceptionModule(256,  128, self.hidden_widths[2],  192, self.hidden_widths[3],  96,  64)
        self.maxpool1 = nn.MaxPool2d(3, stride=2, padding=1)

        self.inc4a = InceptionModule(480,  192, self.hidden_widths[4],  208, self.hidden_widths[5],  48,  64)
        self.inc4b = InceptionModule(512,  160, self.hidden_widths[6],  224, self.hidden_widths[7],  64,  64)
        self.inc4c = InceptionModule(512,  128, self.hidden_widths[8],  256, self.hidden_widths[9],  64,  64)
        self.inc4d = InceptionModule(512,  112, self.hidden_widths[10], 288, self.hidden_widths[11], 64,  64)
        self.inc4e = InceptionModule(528,  256, self.hidden_widths[12], 320, self.hidden_widths[13], 128, 128)
        self.maxpool2 = nn.MaxPool2d(3, stride=2, padding=1)

        self.inc5a = InceptionModule(832,  256, self.hidden_widths[14], 320, self.hidden_widths[15], 128, 128)
        self.inc5b = InceptionModule(832,  384, self.hidden_widths[16], 384, self.hidden_widths[17], 128, 128)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(1024, num_classes)

        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight); nn.init.zeros_(module.bias)

    def get_prunable_blocks(self):
        # Maps perfectly to the exact structure expected by the original pruning logic
        return [
            self.inc3a.branch2, self.inc3a.branch3,
            self.inc3b.branch2, self.inc3b.branch3,
            self.inc4a.branch2, self.inc4a.branch3,
            self.inc4b.branch2, self.inc4b.branch3,
            self.inc4c.branch2, self.inc4c.branch3,
            self.inc4d.branch2, self.inc4d.branch3,
            self.inc4e.branch2, self.inc4e.branch3,
            self.inc5a.branch2, self.inc5a.branch3,
            self.inc5b.branch2, self.inc5b.branch3
        ]

    def forward(self, x):
        x = self.stem(x)
        x = self.inc3a(x); x = self.inc3b(x)
        x = self.maxpool1(x)
        x = self.inc4a(x); x = self.inc4b(x); x = self.inc4c(x); x = self.inc4d(x); x = self.inc4e(x)
        x = self.maxpool2(x)
        x = self.inc5a(x); x = self.inc5b(x)
        return self.fc(self.pool(x).flatten(1))

    def architecture(self):
        return {"hidden_widths": self.hidden_widths, "num_classes": self.num_classes}

def model_payload(model, **extra):
    return {"architecture": model.architecture(), "config": CONFIG,
            "model_state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, **extra}

def load_model(path, device=DEVICE):
    saved = torch.load(path, map_location="cpu", weights_only=False)
    model = GoogLeNetCIFAR(**saved["architecture"])
    model.load_state_dict(saved["model_state_dict"], strict=True)
    return model.to(device), saved

seed_all(CONFIG["seed"])
base_model = GoogLeNetCIFAR().to(DEVICE)
print("Base parameters:", f"{sum(p.numel() for p in base_model.parameters()):,}")
print("Prunable internal widths:", base_model.hidden_widths)

Base parameters: 5,964,164
Prunable internal widths: [96, 16, 128, 32, 96, 16, 112, 24, 128, 24, 144, 32, 160, 32, 160, 32, 192, 48]


## 6. Training, evaluation and resumable checkpoints
Every epoch saves loss, top-1/top-5 accuracy, learning rate and time; the optimizer and random states are saved for resuming.

In [7]:
criterion = nn.CrossEntropyLoss()
def rng_state():
    return {"python":random.getstate(), "numpy":np.random.get_state(),
            "torch":torch.get_rng_state(), "cuda":torch.cuda.get_rng_state_all()}
def restore_rng(state):
    random.setstate(state["python"]); np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"]); torch.cuda.set_rng_state_all(state["cuda"])
def run_epoch(model, loader, optimizer=None, scaler=None, predictions=False):
    training = optimizer is not None
    model.train(training)
    loss_sum,correct1,correct5,total = 0.0,0,0,0
    labels_saved, probabilities = [],[]
    with torch.set_grad_enabled(training):
        for images,labels in loader:
            images,labels = images.to(DEVICE,non_blocking=True),labels.to(DEVICE,non_blocking=True)
            if training: optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda",enabled=AMP):
                logits=model(images); loss=criterion(logits,labels)
            if not torch.isfinite(loss): raise FloatingPointError("Non-finite loss; last checkpoint is safe.")
            if training:
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            n=labels.size(0); top=logits.detach().topk(5,dim=1).indices
            loss_sum+=loss.item()*n; total+=n
            correct1+=(top[:,0]==labels).sum().item()
            correct5+=(top==labels[:,None]).any(dim=1).sum().item()
            if predictions:
                labels_saved.append(labels.cpu().numpy())
                probabilities.append(logits.detach().float().softmax(dim=1).cpu().numpy())
    metrics={"loss":loss_sum/total,"top1":100*correct1/total,"top5":100*correct5/total,"images":total}
    return (metrics,np.concatenate(labels_saved),np.concatenate(probabilities)) if predictions else metrics
def learning_rate(epoch,epochs,initial,warmup):
    if epoch<=warmup: return initial*epoch/warmup
    fraction=(epoch-warmup-1)/max(1,epochs-warmup-1)
    return initial*(0.01+0.99*0.5*(1+math.cos(math.pi*fraction)))
def train_stage(model,stage,epochs,initial_lr,warmup=0):
    folder=RUN_DIR/stage; folder.mkdir(exist_ok=True)
    best_path,last_path=folder/"best.pt",folder/"last.pt"
    model=model.to(DEVICE)
    optimizer=torch.optim.SGD(model.parameters(),lr=initial_lr,momentum=CONFIG["momentum"],weight_decay=CONFIG["weight_decay"])
    scaler=torch.amp.GradScaler("cuda",enabled=AMP)
    history,start,best_score,best_epoch=[],1,-1.0,0
    if CONFIG["resume"] and last_path.exists():
        saved=torch.load(last_path,map_location="cpu",weights_only=False)
        assert saved["config"]==CONFIG and saved["architecture"]==model.architecture()
        model.load_state_dict(saved["model_state_dict"])
        optimizer.load_state_dict(saved["optimizer_state_dict"]); scaler.load_state_dict(saved["scaler_state_dict"])
        history,start=saved["history"],saved["epoch"]+1
        best_score,best_epoch=saved["best_validation_top1"],saved["best_epoch"]
        restore_rng(saved["rng_state"])
        print(f"{stage}: resuming after epoch {start-1}.")
    else:
        seed_all(CONFIG["seed"]+(1000 if stage=="finetune" else 0))
        initial=run_epoch(model,val_loader); best_score=initial["top1"]
        save_torch(model_payload(model,epoch=0,validation=initial),best_path)
    for epoch in range(start,epochs+1):
        began=time.perf_counter()
        lr=learning_rate(epoch,epochs,initial_lr,warmup)
        for group in optimizer.param_groups: group["lr"]=lr
        loader=make_loader(TRAIN,True,CONFIG["seed"]+epoch+10000*(stage=="finetune"))
        train=run_epoch(model,loader,optimizer,scaler)
        val=run_epoch(model,val_loader)
        row={"epoch":epoch,"lr":lr,"seconds":time.perf_counter()-began,
             **{f"train_{k}":v for k,v in train.items()},**{f"validation_{k}":v for k,v in val.items()}}
        history.append(row)
        if val["top1"]>best_score:
            best_score,best_epoch=val["top1"],epoch
            save_torch(model_payload(model,epoch=epoch,validation=val),best_path)
        save_torch(model_payload(model,epoch=epoch,history=history,best_epoch=best_epoch,
            best_validation_top1=best_score,optimizer_state_dict=optimizer.state_dict(),
            scaler_state_dict=scaler.state_dict(),rng_state=rng_state()),last_path)
        pd.DataFrame(history).to_csv(folder/"history.csv",index=False)
        write_json({"stage":stage,"epoch":epoch,"epochs":epochs,"validation_top1":val["top1"],"complete":False},RUN_DIR/"status.json")
        print(f"{stage} {epoch:02d}/{epochs} | train {train['top1']:.2f}% | val {val['top1']:.2f}% | "
              f"val loss {val['loss']:.4f} | lr {lr:.5f} | {row['seconds']:.1f}s",flush=True)
    model,saved=load_model(best_path)
    print(f"Selected {stage} epoch {saved['epoch']}; validation {saved['validation']['top1']:.2f}%.")
    return model,history

## 7. Average absolute weights → DCT → NMF → similarity groups
NMF bases are normalized with compensating coefficients before cosine comparisons; unstable NMF layers remain unpruned and are reported.
All pairs in each group must pass both spectral and signed-weight checks; the medoid is retained and ties use the lowest channel index.

In [8]:
EPS=1e-12
def frequency_description(weights):
    weights=np.asarray(weights,dtype=np.float64)
    averaged=np.abs(weights).mean(axis=1)  # Average FIRST: one spatial map per output filter.
    dct=dctn(averaged,type=2,axes=(-2,-1),norm="ortho")
    energy=np.square(dct).reshape(len(weights),-1)
    totals=energy.sum(axis=1)
    valid=totals>EPS
    probability=np.zeros_like(energy)
    probability[valid]=energy[valid]/totals[valid,None]
    return probability,valid,averaged

def nmf_fingerprints(probability):
    rank=min(CONFIG["nmf_rank"],*probability.shape)
    attempts=[]
    for limit in [CONFIG["nmf_max_iter"],CONFIG["nmf_retry_max_iter"]]:
        with warnings.catch_warnings(record=True) as caught,threadpool_limits(limits=2):
            warnings.simplefilter("always",ConvergenceWarning)
            nmf=NMF(n_components=rank,init="nndsvda",random_state=CONFIG["seed"],
                    max_iter=limit,tol=CONFIG["nmf_tolerance"])
            mixing=nmf.fit_transform(probability)
        converged=not any(issubclass(w.category,ConvergenceWarning) for w in caught)
        attempts.append({"max_iter":limit,"iterations":int(nmf.n_iter_),"converged":converged})
        if converged: break
    bases=nmf.components_.copy()
    scale=np.maximum(np.linalg.norm(bases,axis=1),EPS)
    bases/=scale[:,None]; mixing*=scale[None,:]
    error=float(np.linalg.norm(probability-mixing@bases)/max(np.linalg.norm(probability),EPS))
    fingerprints=mixing/np.maximum(np.linalg.norm(mixing,axis=1,keepdims=True),EPS)
    return fingerprints,bases,{"rank":rank,"converged":converged,"relative_reconstruction_error":error,"attempts":attempts}

def group_channels(similarity,eligible):
    n=len(similarity)
    if n==1: return [[0]],[0]
    distance=np.where(eligible,np.clip(1-similarity,0,2),2.0)
    distance=(distance+distance.T)/2; np.fill_diagonal(distance,0)
    hierarchy=linkage(squareform(distance,checks=True),method="complete")
    labels=fcluster(hierarchy,t=1-CONFIG["similarity_threshold"]+1e-12,criterion="distance")
    groups=[np.flatnonzero(labels==label).tolist() for label in np.unique(labels)]
    groups.sort(key=lambda group:min(group))
    representatives=[]
    for group in groups:
        within=similarity[np.ix_(group,group)]
        representatives.append(group[int(np.argmin((1-within).sum(axis=1)))])
        for a in group:
            for b in group:
                assert a==b or (eligible[a,b] and similarity[a,b]>=CONFIG["similarity_threshold"]-1e-10)
    return groups,representatives

def analyse_filters(weights,bn):
    raw=np.asarray(weights,dtype=np.float64)
    probability,valid,averaged=frequency_description(raw)
    n=len(raw)
    if valid.sum()<CONFIG["nmf_rank"]:
        groups=[[i] for i in range(n)]
        return list(range(n)),groups,[],{"converged":False,"reason":"Too few nonzero filters","original_channels":n,
            "kept_channels":n,"removed_channels":0,"spectral_pairs":0,"eligible_pairs":0}, {"probability":probability}
    fingerprints,bases,diagnostics=nmf_fingerprints(probability)
    similarity=np.clip(fingerprints@fingerprints.T,-1,1)
    factor=bn.weight.detach().cpu().double().numpy()/np.sqrt(bn.running_var.detach().cpu().double().numpy()+bn.eps)
    bias=bn.bias.detach().cpu().double().numpy()-factor*bn.running_mean.detach().cpu().double().numpy()
    effective=(raw*factor[:,None,None,None]).reshape(n,-1)
    norms=np.linalg.norm(effective,axis=1)
    unit=effective/np.maximum(norms[:,None],EPS)
    signed=np.clip(unit@unit.T,-1,1)
    relative_bias=np.abs(bias[:,None]-bias[None,:])/np.maximum((norms[:,None]+norms[None,:])/2,EPS)
    valid=valid & (norms>EPS) & (np.linalg.norm(fingerprints,axis=1)>EPS)
    spectral=(similarity>=CONFIG["similarity_threshold"]) & valid[:,None] & valid[None,:]
    eligible=spectral & (signed>=CONFIG["signed_cosine_min"]) & (relative_bias<=CONFIG["relative_bias_max"])
    if not diagnostics["converged"]: eligible[:]=False
    groups,representatives=group_channels(similarity,eligible)
    kept=sorted(representatives)
    decisions=[]
    for group,representative in zip(groups,representatives):
        for candidate in group:
            if candidate!=representative:
                decisions.append({"removed":candidate,"representative":representative,"group_size":len(group),
                    "nmf_cosine_similarity":float(similarity[candidate,representative]),
                    "signed_cosine":float(signed[candidate,representative]),
                    "relative_bias_difference":float(relative_bias[candidate,representative])})
    assert all(d["representative"] in kept and d["removed"] not in kept for d in decisions)
    diagnostics.update({"original_channels":n,"kept_channels":len(kept),"removed_channels":n-len(kept),
        "spectral_pairs":int(np.triu(spectral,1).sum()),"eligible_pairs":int(np.triu(eligible,1).sum()),
        "invalid_filters_retained":int((~valid).sum()),"removal_percent":100*(n-len(kept))/n})
    return kept,groups,decisions,diagnostics,{"averaged_abs_weights":averaged,"probability":probability,
        "fingerprints":fingerprints,"bases":bases,"nmf_similarity":similarity,"signed_similarity":signed}

def build_plan(model):
    plan,rows,all_groups,all_decisions,all_diagnostics={},[],{},[],{}
    folder=RUN_DIR/"spectral_diagnostics"; folder.mkdir(exist_ok=True)
    # We iterate over the 18 specific prunable reduction branches across the Inception modules
    for index,block in enumerate(model.get_prunable_blocks()):
        weights=block.conv1.weight.detach().cpu().numpy()
        kept,groups,decisions,diagnostics,arrays=analyse_filters(weights,block.bn1)
        plan[str(index)]=kept; all_groups[str(index)]=groups; all_diagnostics[str(index)]=diagnostics
        row={"block":index+1,**{k:v for k,v in diagnostics.items() if k!="attempts"}}
        rows.append(row)
        all_decisions.extend({"block":index+1,**d} for d in decisions)
        np.savez_compressed(folder/f"block_{index+1}.npz",**arrays)
        print(f"Block {index+1}: {weights.shape[0]} -> {len(kept)} channels; NMF converged: {diagnostics['converged']}")
    write_json(plan,RUN_DIR/"kept_channels.json"); write_json(all_groups,RUN_DIR/"groups.json")
    write_json(all_diagnostics,RUN_DIR/"nmf_diagnostics.json")
    write_json(all_decisions,RUN_DIR/"pruning_decisions.json")
    pd.DataFrame(rows).to_csv(RUN_DIR/"layer_pruning.csv",index=False)
    return plan,pd.DataFrame(rows)

## 8. Remove channels and test the mathematics before training
Tests cover the exact averaging order, a sign-reversed filter, a similarity chain, residual connections, saved architecture and a real gradient update.

In [9]:
def make_pruned_model(original, plan):
    # original contains 18 mapped prunable branch widths
    small = GoogLeNetCIFAR([len(plan[str(i)]) for i in range(18)], original.num_classes)
    state = {k: v.detach().cpu().clone() for k, v in original.state_dict().items()}

    # Map index 0..17 to the actual module prefixes in the state_dict
    prefixes = [
        "inc3a.branch2.", "inc3a.branch3.",
        "inc3b.branch2.", "inc3b.branch3.",
        "inc4a.branch2.", "inc4a.branch3.",
        "inc4b.branch2.", "inc4b.branch3.",
        "inc4c.branch2.", "inc4c.branch3.",
        "inc4d.branch2.", "inc4d.branch3.",
        "inc4e.branch2.", "inc4e.branch3.",
        "inc5a.branch2.", "inc5a.branch3.",
        "inc5b.branch2.", "inc5b.branch3."
    ]

    for index, prefix in enumerate(prefixes):
        keep = torch.tensor(plan[str(index)], dtype=torch.long)
        state[prefix+"conv1.weight"] = state[prefix+"conv1.weight"][keep]
        for name in ["weight", "bias", "running_mean", "running_var"]:
            state[prefix+"bn1."+name] = state[prefix+"bn1."+name][keep]
        state[prefix+"conv2.weight"] = state[prefix+"conv2.weight"][:, keep]

    small.load_state_dict(state, strict=True)
    return small.to(DEVICE)

@torch.no_grad()
def check_surgery(original, small, plan, sample):
    original.eval(); small.eval(); handles = []
    for index, block in enumerate(original.get_prunable_blocks()):
        mask = torch.zeros(1, block.conv1.out_channels, 1, 1, device=DEVICE)
        mask[:, plan[str(index)]] = 1
        handles.append(block.relu1.register_forward_hook(lambda module, args, out, mask=mask: out * mask))
    try:
        expected = original(sample.to(DEVICE)); actual = small(sample.to(DEVICE))
        torch.testing.assert_close(actual, expected, rtol=3e-4, atol=3e-5)
        error = float((actual-expected).abs().max())
    finally:
        for handle in handles: handle.remove()
    return error

def small_checks():
    state=rng_state()
    try:
        seed_all(123)
        weights=np.random.default_rng(123).normal(size=(12,4,3,3))
        weights[1]=weights[0]; weights[2]=-weights[0]
        p,valid,averaged=frequency_description(weights)
        assert np.allclose(averaged,np.abs(weights).mean(axis=1))
        assert np.allclose(p.sum(axis=1),1) and np.allclose(p[0],p[2])
        assert np.allclose((dctn(averaged,axes=(-2,-1),norm="ortho")**2).sum(),(averaged**2).sum())

        kept,groups,decisions,diag,_=analyse_filters(weights,nn.BatchNorm2d(12))
        assert diag["converged"]
        assert any(0 in g and 1 in g for g in groups)
        assert not any(0 in g and 2 in g for g in groups)

        sim=np.array([[1,.96,.90],[.96,1,.96],[.90,.96,1]])
        chain_groups,_=group_channels(sim,sim>=CONFIG["similarity_threshold"])
        assert max(map(len,chain_groups))==2

        tiny=GoogLeNetCIFAR().to(DEVICE)
        with torch.no_grad():
            for m in tiny.modules():
                if isinstance(m,nn.BatchNorm2d):
                    m.weight.uniform_(.5,1.5); m.bias.uniform_(-.2,.2)
                    m.running_mean.uniform_(-.3,.3); m.running_var.uniform_(.5,1.5)

        plan={str(i):list(range(0,b.conv1.out_channels,2)) for i,b in enumerate(tiny.get_prunable_blocks())}
        small=make_pruned_model(tiny,plan)
        error=check_surgery(tiny,small,plan,torch.randn(4,3,32,32,device=DEVICE))

        payload=model_payload(small); rebuilt=GoogLeNetCIFAR(**payload["architecture"])
        rebuilt.load_state_dict(payload["model_state_dict"],strict=True)
        small.train(); before=small.get_prunable_blocks()[0].conv1.weight.detach().clone()
        opt=torch.optim.SGD(small.parameters(),lr=.01)
        loss=criterion(small(torch.randn(4,3,32,32,device=DEVICE)),torch.arange(4,device=DEVICE))
        opt.zero_grad(); loss.backward(); opt.step()
        assert not torch.equal(before,small.get_prunable_blocks()[0].conv1.weight)

        result={"passed":True,"surgery_max_error":error,"checks":["average before DCT","DCT energy", "positive and negative copies",
                 "no transitive grouping","residual channel removal","reload","gradient update"]}
        write_json(result,RUN_DIR/"implementation_checks.json")
        print("All small implementation checks passed.",result)
    finally:
        restore_rng(state)
small_checks()

All small implementation checks passed. {'passed': True, 'surgery_max_error': 6.103515625e-05, 'checks': ['average before DCT', 'DCT energy', 'positive and negative copies', 'no transitive grouping', 'residual channel removal', 'reload', 'gradient update']}


## 9. Train and save the base model for 50 epochs
The best and last checkpoints are both saved; the best validation checkpoint is used for pruning.

In [10]:
base_model,base_history=train_stage(base_model,"base",CONFIG["base_epochs"],CONFIG["base_lr"],CONFIG["warmup_epochs"])
_,base_saved=load_model(RUN_DIR/"base/best.pt",device="cpu")
save_torch(base_saved,RUN_DIR/"base_model.pt")
base_validation=run_epoch(base_model,val_loader)
write_json(base_validation,RUN_DIR/"base_validation.json")
print("BASE MODEL SAVED:",RUN_DIR/"base_model.pt")
print("Base validation:",base_validation)

base 01/50 | train 12.33% | val 15.86% | val loss 3.4836 | lr 0.02000 | 79.5s
base 02/50 | train 24.74% | val 27.98% | val loss 2.8488 | lr 0.04000 | 80.9s
base 03/50 | train 36.12% | val 40.18% | val loss 2.2484 | lr 0.06000 | 80.9s
base 04/50 | train 45.12% | val 44.22% | val loss 2.0958 | lr 0.08000 | 81.0s
base 05/50 | train 50.47% | val 43.20% | val loss 2.1628 | lr 0.10000 | 81.0s
base 06/50 | train 55.62% | val 49.14% | val loss 1.8970 | lr 0.10000 | 80.8s
base 07/50 | train 58.82% | val 49.86% | val loss 1.9176 | lr 0.09987 | 80.8s
base 08/50 | train 60.96% | val 50.94% | val loss 1.7929 | lr 0.09950 | 81.1s
base 09/50 | train 62.32% | val 44.78% | val loss 2.2239 | lr 0.09887 | 81.0s
base 10/50 | train 63.26% | val 45.88% | val loss 2.1174 | lr 0.09799 | 80.9s
base 11/50 | train 64.23% | val 48.88% | val loss 2.0910 | lr 0.09688 | 80.8s
base 12/50 | train 65.43% | val 51.16% | val loss 1.8806 | lr 0.09553 | 80.9s
base 13/50 | train 66.07% | val 55.22% | val loss 1.6823 | lr 0.

## 10. Prune by similarity and verify the actual trained network
There is no fixed removal percentage; the table reports what the 0.95 threshold and signed checks actually remove.

In [11]:
pruned_path=RUN_DIR/"pruned_before_finetune.pt"
if pruned_path.exists() and CONFIG["resume"]:
    plan=json.loads((RUN_DIR/"kept_channels.json").read_text())
    layer_results=pd.read_csv(RUN_DIR/"layer_pruning.csv")
    pruned_model,_=load_model(pruned_path)
else:
    plan,layer_results=build_plan(base_model)
    pruned_model=make_pruned_model(base_model,plan)
    sample=torch.stack([TRAIN_EVAL[i][0] for i in range(4)]).to(DEVICE)
    surgery_error=check_surgery(base_model,pruned_model,plan,sample)
    save_torch(model_payload(pruned_model,stage="pruned_before_finetune",surgery_max_error=surgery_error),pruned_path)
    print("Trained network removal check passed; maximum output difference:",surgery_error)
pruned_validation=run_epoch(pruned_model,val_loader)
write_json(pruned_validation,RUN_DIR/"pruned_validation.json")
display(layer_results[["block","original_channels","kept_channels","removed_channels","spectral_pairs","eligible_pairs","converged"]])
print("Validation before pruning:",base_validation["top1"])
print("Validation after pruning:",pruned_validation["top1"])
removed=int(layer_results.removed_channels.sum())
print("Channels removed:",removed,"out of",int(layer_results.original_channels.sum()),"eligible internal channels.")
if removed==0: print("No channels were removed; this run cannot demonstrate compression.")
base_model=base_model.cpu(); torch.cuda.empty_cache()

Block 1: 96 -> 96 channels; NMF converged: False
Block 2: 16 -> 10 channels; NMF converged: True
Block 3: 128 -> 34 channels; NMF converged: True
Block 4: 32 -> 32 channels; NMF converged: False
Block 5: 96 -> 96 channels; NMF converged: False
Block 6: 16 -> 13 channels; NMF converged: True
Block 7: 112 -> 50 channels; NMF converged: True
Block 8: 24 -> 13 channels; NMF converged: True
Block 9: 128 -> 61 channels; NMF converged: True
Block 10: 24 -> 14 channels; NMF converged: True
Block 11: 144 -> 67 channels; NMF converged: True
Block 12: 32 -> 32 channels; NMF converged: False
Block 13: 160 -> 86 channels; NMF converged: True
Block 14: 32 -> 32 channels; NMF converged: False
Block 15: 160 -> 97 channels; NMF converged: True
Block 16: 32 -> 32 channels; NMF converged: False
Block 17: 192 -> 192 channels; NMF converged: False
Block 18: 48 -> 23 channels; NMF converged: True
Trained network removal check passed; maximum output difference: 1.1920928955078125e-06


,block,original_channels,kept_channels,removed_channels,spectral_pairs,eligible_pairs,converged
0,1,96,96,0,4560,0,False
1,2,16,10,6,120,18,True
2,3,128,34,94,8128,1624,True
3,4,32,32,0,496,0,False
4,5,96,96,0,4560,0,False
5,6,16,13,3,120,6,True
6,7,112,50,62,6216,474,True
7,8,24,13,11,276,41,True
8,9,128,61,67,8128,412,True
9,10,24,14,10,276,32,True


Validation before pruning: 78.28
Validation after pruning: 6.64
Channels removed: 492 out of 1472 eligible internal channels.


## 11. Measure the effect of refreshing BatchNorm statistics
This diagnostic uses 5,000 training images without gradient updates; fine-tuning below still starts from the original pruned checkpoint.

In [12]:
@torch.no_grad()
def recalibrate_bn(model,loader):
    model.eval(); changed=[]
    for module in model.modules():
        if isinstance(module,nn.BatchNorm2d):
            changed.append((module,module.momentum))
            module.reset_running_stats(); module.momentum=None; module.train()
    try:
        for images,_ in loader:
            with torch.amp.autocast(device_type="cuda",enabled=AMP): model(images.to(DEVICE))
    finally:
        for module,momentum in changed: module.momentum=momentum
        model.eval()
bn_model=copy.deepcopy(pruned_model)
recalibrate_bn(bn_model,calibration_loader)
bn_validation=run_epoch(bn_model,val_loader)
save_torch(model_payload(bn_model,stage="bn_recalibrated",validation=bn_validation),RUN_DIR/"pruned_bn_recalibrated.pt")
write_json(bn_validation,RUN_DIR/"bn_recalibrated_validation.json")
print("Validation before BN refresh:",pruned_validation["top1"],"| after:",bn_validation["top1"])
del bn_model; torch.cuda.empty_cache()

Validation before BN refresh: 6.64 | after: 62.5


## 12. Fine-tune for 50 epochs and save the final model
Surviving learned weights continue training with a fresh optimizer and a smaller learning rate.

In [13]:
pruned_model,finetune_history=train_stage(pruned_model,"finetune",CONFIG["finetune_epochs"],CONFIG["finetune_lr"])
_,final_saved=load_model(RUN_DIR/"finetune/best.pt",device="cpu")
save_torch(final_saved,RUN_DIR/"finetuned_model.pt")
final_validation=run_epoch(pruned_model,val_loader)
print("FINE-TUNED MODEL SAVED:",RUN_DIR/"finetuned_model.pt")
print("Final validation:",final_validation)

finetune 01/50 | train 95.88% | val 72.24% | val loss 1.1142 | lr 0.01000 | 75.1s
finetune 02/50 | train 97.94% | val 72.86% | val loss 1.0624 | lr 0.00999 | 75.8s
finetune 03/50 | train 98.62% | val 74.14% | val loss 1.0382 | lr 0.00996 | 74.5s
finetune 04/50 | train 98.93% | val 73.84% | val loss 1.0272 | lr 0.00991 | 74.2s
finetune 05/50 | train 99.13% | val 73.96% | val loss 1.0811 | lr 0.00984 | 74.1s
finetune 06/50 | train 99.30% | val 75.20% | val loss 1.0072 | lr 0.00975 | 74.7s
finetune 07/50 | train 99.50% | val 73.96% | val loss 1.0440 | lr 0.00964 | 74.5s
finetune 08/50 | train 99.52% | val 74.88% | val loss 1.0579 | lr 0.00951 | 74.5s
finetune 09/50 | train 99.58% | val 75.48% | val loss 0.9977 | lr 0.00936 | 74.5s
finetune 10/50 | train 99.69% | val 75.56% | val loss 0.9983 | lr 0.00920 | 74.8s
finetune 11/50 | train 99.74% | val 75.58% | val loss 0.9892 | lr 0.00902 | 74.5s
finetune 12/50 | train 99.79% | val 75.44% | val loss 0.9913 | lr 0.00882 | 74.3s
finetune 13/50 |

## 13. Final accuracy, per-class results and confidence quality
All models use the same official test images; test predictions are never used to choose checkpoints or pruning settings.
ECE measures confidence mismatch (lower is better); its value depends on the 15 fixed confidence bins.

In [14]:
test_data=torchvision.datasets.CIFAR100(data_root,train=False,download=False,transform=eval_transform)
test_loader=make_loader(test_data)
train_eval_loader=make_loader(TRAIN_EVAL)
def confidence_metrics(labels,probs):
    predicted=probs.argmax(axis=1); correct=predicted==labels; confidence=probs.max(axis=1)
    ece=0.0
    for index in range(15):
        selected=(confidence>=index/15)&((confidence<(index+1)/15) if index<14 else (confidence<=1))
        if selected.any(): ece+=selected.mean()*abs(correct[selected].mean()-confidence[selected].mean())
    one_hot=np.eye(probs.shape[1])[labels]
    p=correct.mean(); n=len(labels); z=1.96
    centre=(p+z*z/(2*n))/(1+z*z/n)
    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/(1+z*z/n)
    return {"ece_percent":100*float(ece),"brier_score":float(np.square(probs-one_hot).sum(axis=1).mean()),
            "accuracy_ci95_low":100*(centre-half),"accuracy_ci95_high":100*(centre+half)}
def resources(model):
    macs=[0]; hooks=[]
    def count(module,args,out):
        if isinstance(module,nn.Conv2d):
            macs[0]+=out.numel()*(module.in_channels//module.groups)*math.prod(module.kernel_size)
        elif isinstance(module,nn.Linear): macs[0]+=out.numel()*module.in_features
    for m in model.modules():
        if isinstance(m,(nn.Conv2d,nn.Linear)): hooks.append(m.register_forward_hook(count))
    try:
        model.eval()
        with torch.no_grad(): model(torch.zeros(1,3,32,32,device=DEVICE))
    finally:
        for h in hooks: h.remove()
    return {"parameters":sum(p.numel() for p in model.parameters()),"macs":int(macs[0])}
model_files={"base":RUN_DIR/"base_model.pt","pruned":pruned_path,
             "bn_recalibrated":RUN_DIR/"pruned_bn_recalibrated.pt","finetuned":RUN_DIR/"finetuned_model.pt"}
pruned_model=pruned_model.cpu(); torch.cuda.empty_cache()
rows,per_class,test_predictions=[],{},{}
for stage,path in model_files.items():
    model,saved=load_model(path)
    metrics,labels,probs=run_epoch(model,test_loader,predictions=True)
    val=run_epoch(model,val_loader); train=run_epoch(model,train_eval_loader)
    predicted=probs.argmax(axis=1); test_predictions[stage]=(labels,predicted)
    confusion=np.zeros((100,100),dtype=np.int64); np.add.at(confusion,(labels,predicted),1)
    per_class[stage]=100*confusion.diagonal()/confusion.sum(axis=1)
    pd.DataFrame(confusion,index=CLASSES,columns=CLASSES).to_csv(RUN_DIR/f"{stage}_confusion.csv")
    np.savez_compressed(RUN_DIR/f"{stage}_predictions.npz",labels=labels,predictions=predicted,probabilities=probs)
    pd.DataFrame({"test_index":np.arange(len(labels)),"label":labels,"prediction":predicted,
                  "confidence":probs.max(axis=1)}).to_csv(RUN_DIR/f"{stage}_predictions.csv",index=False)
    rows.append({"stage":stage,"train_eval_top1":train["top1"],"validation_top1":val["top1"],
                 "test_top1":metrics["top1"],"test_top5":metrics["top5"],"test_loss":metrics["loss"],
                 **confidence_metrics(labels,probs),**resources(model),"model_bytes":path.stat().st_size,
                 "selected_epoch":saved.get("epoch")})
    del model; torch.cuda.empty_cache()
results=pd.DataFrame(rows); results.to_csv(RUN_DIR/"results.csv",index=False)
class_results=pd.DataFrame({"class":CLASSES,**per_class})
class_results["finetuned_minus_base_pp"]=class_results.finetuned-class_results.base
class_results.to_csv(RUN_DIR/"per_class_accuracy.csv",index=False)
display(results.round(4)); display(class_results.round(2))
# Paired uncertainty uses the same test examples; it does not include training-seed variation.
y,base_pred=test_predictions["base"]; y2,final_pred=test_predictions["finetuned"]
assert np.array_equal(y,y2)
base_ok,final_ok=base_pred==y,final_pred==y
delta=final_ok.astype(float)-base_ok.astype(float)
se=float(delta.std(ddof=1)/np.sqrt(len(delta)))
lost=int((base_ok & ~final_ok).sum()); gained=int((~base_ok & final_ok).sum())
paired={"net_accuracy_change_pp":100*float(delta.mean()),"approx_ci95_low_pp":100*(float(delta.mean())-1.96*se),
        "approx_ci95_high_pp":100*(float(delta.mean())+1.96*se),"previously_correct_now_wrong":lost,
        "previously_wrong_now_correct":gained,"mcnemar_exact_p":float(binomtest(lost,lost+gained,.5).pvalue) if lost+gained else 1.0,
        "scope":"Paired test-set uncertainty only; one training seed."}
write_json(paired,RUN_DIR/"paired_comparison.json"); print("Paired comparison:",paired)

,stage,train_eval_top1,validation_top1,test_top1,test_top5,test_loss,ece_percent,brier_score,accuracy_ci95_low,accuracy_ci95_high,parameters,macs,model_bytes,selected_epoch
0,base,99.9800,78.28,78.21,94.68,0.8393,6.6912,0.3146,77.3901,79.0082,5964164,1447333888,24016851,48.0
1,pruned,6.9711,6.64,6.55,21.59,5.1808,35.0840,1.1603,6.0816,7.0518,4550076,1002033152,18360555,NaN
2,bn_recalibrated,84.1467,62.50,62.41,86.22,1.5216,8.4880,0.5130,61.4561,63.3544,4550076,1002033152,18360619,NaN
3,finetuned,99.9867,77.84,77.66,94.20,0.8595,4.7212,0.3183,76.8331,78.4657,4550076,1002033152,18358061,47.0


,class,base,pruned,bn_recalibrated,finetuned,finetuned_minus_base_pp
0,apple,92.0,0.0,87.0,95.0,3.0
1,aquarium_fish,93.0,0.0,58.0,93.0,0.0
2,baby,71.0,0.0,62.0,69.0,-2.0
3,bear,65.0,0.0,38.0,66.0,1.0
4,beaver,68.0,0.0,42.0,67.0,-1.0
...,...,...,...,...,...,...
95,whale,72.0,0.0,64.0,75.0,3.0
96,willow_tree,72.0,11.0,63.0,68.0,-4.0
97,wolf,88.0,0.0,66.0,87.0,-1.0
98,woman,66.0,0.0,42.0,63.0,-3.0


Paired comparison: {'net_accuracy_change_pp': -0.5499999999999999, 'approx_ci95_low_pp': -1.0969683254402431, 'approx_ci95_high_pp': -0.0030316745597569646, 'previously_correct_now_wrong': 417, 'previously_wrong_now_correct': 362, 'mcnemar_exact_p': 0.05295112809145511, 'scope': 'Paired test-set uncertainty only; one training seed.'}


## 14. Measure real inference speed and GPU memory
Measure both models on the same GPU, inputs, precision and backend settings; alternate their order across five repeats.
Timing includes model inference only, with input already on the GPU; MACs above count Conv/Linear work only.

In [15]:
@torch.inference_mode()
def measure_once(model,sample,iterations=100):
    model.eval()
    with torch.amp.autocast(device_type="cuda",enabled=AMP):
        for _ in range(30): model(sample)
        torch.cuda.synchronize()
        start,end=torch.cuda.Event(enable_timing=True),torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(iterations): model(sample)
        end.record(); end.synchronize()
    return float(start.elapsed_time(end)/iterations)
speed_models={s:load_model(model_files[s])[0] for s in ["base","finetuned"]}
raw_times=[]
for batch in [1,128]:
    sample=torch.randn(batch,3,32,32,device=DEVICE)
    for repeat in range(5):
        order=["base","finetuned"] if repeat%2==0 else ["finetuned","base"]
        for stage in order:
            raw_times.append({"stage":stage,"batch_size":batch,"repeat":repeat,
                              "latency_ms":measure_once(speed_models[stage],sample)})
del speed_models; torch.cuda.empty_cache()
times=pd.DataFrame(raw_times); times.to_csv(RUN_DIR/"latency_repeats.csv",index=False)
speed_rows=[]
for (stage,batch),group in times.groupby(["stage","batch_size"]):
    median=float(group.latency_ms.median())
    speed_rows.append({"stage":stage,"batch_size":int(batch),"median_ms":median,
        "min_ms":float(group.latency_ms.min()),"max_ms":float(group.latency_ms.max()),
        "images_per_second":batch*1000/median,"precision":"autocast_fp16" if AMP else "fp32"})
speeds=pd.DataFrame(speed_rows); speeds.to_csv(RUN_DIR/"latency.csv",index=False); display(speeds.round(4))
memory_rows=[]
for stage in ["base","finetuned"]:
    model,_=load_model(model_files[stage]); model.eval()
    sample=torch.randn(128,3,32,32,device=DEVICE)
    with torch.inference_mode(),torch.amp.autocast(device_type="cuda",enabled=AMP):
        model(sample); torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        before=torch.cuda.memory_allocated()
        model(sample); torch.cuda.synchronize()
        peak=torch.cuda.max_memory_allocated()
    memory_rows.append({"stage":stage,"batch_size":128,"allocated_before_mb":before/1e6,
                        "peak_allocated_mb":peak/1e6,"incremental_peak_mb":(peak-before)/1e6})
    del model,sample; torch.cuda.empty_cache()
pd.DataFrame(memory_rows).to_csv(RUN_DIR/"gpu_memory.csv",index=False)
display(pd.DataFrame(memory_rows).round(2))

,stage,batch_size,median_ms,min_ms,max_ms,images_per_second,precision
0,base,1,9.3066,8.8595,18.6481,107.4510,autocast_fp16
1,base,128,72.0872,70.9852,74.5006,1775.6263,autocast_fp16
2,finetuned,1,10.4608,9.3105,12.2844,95.5948,autocast_fp16
3,finetuned,128,66.1677,65.4256,68.5777,1934.4800,autocast_fp16


,stage,batch_size,allocated_before_mb,peak_allocated_mb,incremental_peak_mb
0,base,128,93.83,605.53,511.71
1,finetuned,128,88.06,599.77,511.71
